# 🔷 STEP 1: DATA LOADING

Load NASA CMAPSS FD001-FD004 datasets, clean column structure, and combine into unified dataset

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import pandas as pd

spark = SparkSession.builder.appName("CMAPSS_RUL").getOrCreate()

# Load all 4 datasets using pandas
base_path = "/Workspace/Users/csd25018@tezu.ac.in/CMAPSS_RUL_Project/"

fd001_pd = pd.read_csv(base_path + "Train FD001.txt", sep=" ", header=None)
fd002_pd = pd.read_csv(base_path + "Train FD002.txt", sep=" ", header=None)
fd003_pd = pd.read_csv(base_path + "Train FD003.txt", sep=" ", header=None)
fd004_pd = pd.read_csv(base_path + "Train FD004.txt", sep=" ", header=None)

print("✅ All datasets loaded successfully")
print(f"FD001 shape: {fd001_pd.shape[0]} rows, {fd001_pd.shape[1]} columns")
print(f"FD002 shape: {fd002_pd.shape[0]} rows, {fd002_pd.shape[1]} columns")
print(f"FD003 shape: {fd003_pd.shape[0]} rows, {fd003_pd.shape[1]} columns")
print(f"FD004 shape: {fd004_pd.shape[0]} rows, {fd004_pd.shape[1]} columns")

✅ All datasets loaded successfully
FD001 shape: 20631 rows, 28 columns
FD002 shape: 53759 rows, 28 columns
FD003 shape: 24720 rows, 28 columns
FD004 shape: 61249 rows, 28 columns


## Remove Extra Columns

The datasets have trailing empty columns (_c26, _c27) that need to be removed

In [0]:
# Remove extra trailing columns (last 2 columns are empty from space-delimited format)
fd001_pd = fd001_pd.iloc[:, :-2]
fd002_pd = fd002_pd.iloc[:, :-2]
fd003_pd = fd003_pd.iloc[:, :-2]
fd004_pd = fd004_pd.iloc[:, :-2]

print("✅ Extra columns removed")
print(f"Updated column count: {fd001_pd.shape[1]}")

✅ Extra columns removed
Updated column count: 26


## Rename Columns with Meaningful Names

Assign proper names: engine_id, cycle, operating conditions (op1-op3), and sensors (s1-s21)

In [0]:
# Define meaningful column names
columns = ["engine_id", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]

# Apply to pandas dataframes first
fd001_pd.columns = columns
fd002_pd.columns = columns
fd003_pd.columns = columns
fd004_pd.columns = columns

# Convert to Spark DataFrames
fd001 = spark.createDataFrame(fd001_pd)
fd002 = spark.createDataFrame(fd002_pd)
fd003 = spark.createDataFrame(fd003_pd)
fd004 = spark.createDataFrame(fd004_pd)

print("✅ Columns renamed successfully")
print("Column names:", columns[:10], "...")

✅ Columns renamed successfully
Column names: ['engine_id', 'cycle', 'op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5'] ...


## Add Dataset Labels

Add a column to identify which dataset each row belongs to (FD001-FD004)

In [0]:
# Add dataset identifier column
fd001 = fd001.withColumn("dataset", lit("FD001"))
fd002 = fd002.withColumn("dataset", lit("FD002"))
fd003 = fd003.withColumn("dataset", lit("FD003"))
fd004 = fd004.withColumn("dataset", lit("FD004"))

print("✅ Dataset labels added")

✅ Dataset labels added


## Combine All Datasets

Merge all 4 datasets into one unified dataframe for comprehensive analysis

In [0]:
# Union all datasets
engine_df = fd001.union(fd002).union(fd003).union(fd004)

print("✅ All datasets combined")
print(f"Total rows: {engine_df.count()}")
print(f"Total columns: {len(engine_df.columns)}")
print(f"Unique engines: {engine_df.select('engine_id').distinct().count()}")

# Display sample
display(engine_df.limit(10))

# Save as Delta table for next notebooks
engine_df.write.format("delta").mode("overwrite").saveAsTable("cmapss_raw_data")
print("\n✅ Data saved as Delta table: cmapss_raw_data")

✅ All datasets combined
Total rows: 160359
Total columns: 27
Unique engines: 260


engine_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,dataset
1,1,-7.0E-4,-4.0E-4,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.419,FD001
1,2,0.0019,-3.0E-4,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.0,23.4236,FD001
1,3,-0.0043,3.0E-4,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,FD001
1,4,7.0E-4,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,FD001
1,5,-0.0019,-2.0E-4,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393,2388,100.0,38.9,23.4044,FD001
1,6,-0.0043,-1.0E-4,100.0,518.67,642.1,1584.47,1398.37,14.62,21.61,554.67,2388.02,9049.68,1.3,47.16,521.68,2388.03,8132.85,8.4108,0.03,391,2388,100.0,38.98,23.3669,FD001
1,7,0.001,1.0E-4,100.0,518.67,642.48,1592.32,1397.77,14.62,21.61,554.34,2388.02,9059.13,1.3,47.36,522.32,2388.03,8132.32,8.3974,0.03,392,2388,100.0,39.1,23.3774,FD001
1,8,-0.0034,3.0E-4,100.0,518.67,642.56,1582.96,1400.97,14.62,21.61,553.85,2388.0,9040.8,1.3,47.24,522.47,2388.03,8131.07,8.4076,0.03,391,2388,100.0,38.97,23.3106,FD001
1,9,8.0E-4,1.0E-4,100.0,518.67,642.12,1590.98,1394.8,14.62,21.61,553.69,2388.05,9046.46,1.3,47.29,521.79,2388.05,8125.69,8.3728,0.03,392,2388,100.0,39.05,23.4066,FD001
1,10,-0.0033,1.0E-4,100.0,518.67,641.71,1591.24,1400.46,14.62,21.61,553.59,2388.05,9051.7,1.3,47.03,521.79,2388.06,8129.38,8.4286,0.03,393,2388,100.0,38.95,23.4694,FD001



✅ Data saved as Delta table: cmapss_raw_data


## 📊 Summary

* Loaded 4 CMAPSS datasets (FD001-FD004)
* Cleaned column structure
* Applied meaningful names
* Combined into unified dataset
* Saved as Delta table: `cmapss_raw_data`

**Next Step**: Proceed to `02_Data_Cleaning` notebook